In [7]:
import pandas as pd
import numpy as np
import joblib
import lightgbm as lgb
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, accuracy_score, silhouette_score

In [8]:
# 1. NẠP DỮ LIỆU
print("📂 Đang nạp dữ liệu cho 2 phương pháp...")
df = pd.read_csv("../data/processed_malicious_url.csv")
X = df.drop(columns=['url', 'target'])
le = LabelEncoder()
y = le.fit_transform(df['target'])

# Chuẩn hóa dữ liệu (Rất quan trọng cho Clustering)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

📂 Đang nạp dữ liệu cho 2 phương pháp...


In [9]:
# --- PHƯƠNG PHÁP 2: CLASSIFICATION (LightGBM) ---
print("\n🚀 Đang huấn luyện LightGBM (Phát cuối cho Accuracy)...")
x_train, x_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.15, stratify=y, random_state=42)


🚀 Đang huấn luyện LightGBM (Phát cuối cho Accuracy)...


In [10]:
lgbm = lgb.LGBMClassifier(
    n_estimators=1500,
    learning_rate=0.05,
    num_leaves=128,          # Tăng leaf để bắt pattern sâu hơn XGBoost

    # Thay thế bằng các tham số chuẩn của LightGBM:
    boosting_type='gbdt',    # Kiểu boosting mặc định, tương đương hist của XGB
    force_row_wise=True,     # Fix cảnh báo [Info] Auto-choosing...
    importance_type='gain',  # Tính độ quan trọng feature theo mức đóng góp

    n_jobs=-1,
    random_state=42
)

lgbm.fit(x_train, y_train, eval_set=[(x_test, y_test)], 
         callbacks=[lgb.early_stopping(stopping_rounds=50), lgb.log_evaluation(100)])


[LightGBM] [Info] Total Bins 1304
[LightGBM] [Info] Number of data points in the train set: 549199, number of used features: 14
[LightGBM] [Info] Start training from score -0.400057
[LightGBM] [Info] Start training from score -1.913864
[LightGBM] [Info] Start training from score -3.307841
[LightGBM] [Info] Start training from score -1.926797
Training until validation scores don't improve for 50 rounds
[100]	valid_0's multi_logloss: 0.410359
[200]	valid_0's multi_logloss: 0.381852
[300]	valid_0's multi_logloss: 0.368598
[400]	valid_0's multi_logloss: 0.362078
[500]	valid_0's multi_logloss: 0.357283
[600]	valid_0's multi_logloss: 0.353833
[700]	valid_0's multi_logloss: 0.351246
[800]	valid_0's multi_logloss: 0.349298
[900]	valid_0's multi_logloss: 0.347907
[1000]	valid_0's multi_logloss: 0.346666
[1100]	valid_0's multi_logloss: 0.345916
[1200]	valid_0's multi_logloss: 0.345459
[1300]	valid_0's multi_logloss: 0.34518
[1400]	valid_0's multi_logloss: 0.344963
[1500]	valid_0's multi_logloss:

,boosting_type,'gbdt'
,num_leaves,128
,max_depth,-1
,learning_rate,0.05
,n_estimators,1500
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [11]:
y_pred = lgbm.predict(x_test)
print(f"\n🎯 Accuracy LightGBM: {accuracy_score(y_test, y_pred)*100:.2f}%")
print(classification_report(y_test, y_pred, target_names=le.classes_))

c:\Users\vppho\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



🎯 Accuracy LightGBM: 86.60%
              precision    recall  f1-score   support

      benign       0.88      0.95      0.91     64962
  defacement       0.86      0.83      0.85     14296
     malware       0.98      0.91      0.94      3547
    phishing       0.75      0.52      0.62     14113

    accuracy                           0.87     96918
   macro avg       0.87      0.80      0.83     96918
weighted avg       0.86      0.87      0.86     96918



In [12]:
feature_names = X.columns.tolist()
joblib.dump(feature_names, "../models/feature_names.pkl")
joblib.dump(lgbm, "../models/lgbm_model.pkl")
joblib.dump(le, "../models/label_encoder.pkl")
joblib.dump(scaler, "../models/scaler.pkl") # CỰC KỲ QUAN TRỌNG
print("💾 Đã lưu model và label encoder thành công!")

💾 Đã lưu model và label encoder thành công!


In [25]:
import re
import math
from urllib.parse import urlparse

# 1. Hàm trích xuất đặc trưng (Phải giống hệt hàm dùng khi Train)
def quick_extract(url):
    u = str(url).strip().lower().replace('[', '').replace(']', '')
    address = u if '://' in u else 'http://' + u
    try: p = urlparse(address)
    except: p = urlparse('http://error-url.com')
    
    hostname = p.netloc.replace('www.', '')
    path = p.path + p.query
    full_url = hostname + path
    
    keywords = ['login', 'verify', 'update', 'secure', 'account', 'banking', 'signin', 'confirm', 'bank']
    trash_tld = ('.tk', '.xyz', '.cc', '.top', '.pw', '.online', '.site', '.biz', '.info')
    popular_tld = ('.com', '.net', '.org', '.vn', '.edu', '.gov')

    feats = {
        'url_len': len(full_url),
        'hostname_len': len(hostname),
        'dot_count': full_url.count('.'),
        'dash_count': hostname.count('-'),
        'digit_ratio': len(re.findall(r'\d', full_url)) / (len(full_url) + 1),
        'entropy': -sum([full_url.count(c)/len(full_url) * math.log2(full_url.count(c)/len(full_url)) for c in set(full_url)]),
        'is_trash_tld': int(hostname.endswith(trash_tld)),
        'is_popular_tld': int(any(hostname.endswith(t) for t in popular_tld)),
        'has_ip': int(bool(re.search(r'(\d{1,3}\.){3}\d{1,3}', hostname))),
        'is_exec': int(bool(re.search(r'\.(exe|apk|msi|bin|js|vbs|scr|zip)$', path))),
        'keyword_count': sum(1 for k in keywords if k in full_url),
        'subdomain_count': len(hostname.split('.')) - 2 if len(hostname.split('.')) > 2 else 0,
        'special_ratio': sum(full_url.count(c) for c in ['-', '.', '_', '@', '?', '&', '=']) / (len(full_url) + 1),
        'has_number_in_host': int(any(char.isdigit() for char in hostname))
    }
    return pd.DataFrame([feats])

# 2. Giao diện Test Nóng
print("🔥 HỆ THỐNG KIỂM TRA URL NHANH 🔥")
input_url = input("Dán link cần test vào đây: ")

# Xử lý & Dự đoán
# Chú ý: Nếu dùng LightGBM thì dùng X_input_scaled, nếu XGBoost thì dùng X_input trực tiếp tùy theo cách ông train
X_input = quick_extract(input_url)

# Nếu Notebook này là LightGBM (cần scale):
X_input_scaled = scaler.transform(X_input)
prob = lgbm.predict_proba(X_input_scaled)

# Nếu Notebook này là XGBoost (ko cần scale):
# prob = model.predict_proba(X_input)

res_idx = np.argmax(prob)
res_label = le.inverse_transform([res_idx])[0]
confidence = np.max(prob) * 100

print(f"\n" + "="*40)
print(f"🔗 URL: {input_url}")
print(f"🛑 Dự đoán: {res_label.upper()}")
print(f"📊 Độ tự tin: {confidence:.2f}%")
print("="*40)

🔥 HỆ THỐNG KIỂM TRA URL NHANH 🔥

🔗 URL: https://pypi.org/project/joblib/
🛑 Dự đoán: BENIGN
📊 Độ tự tin: 54.05%


c:\Users\vppho\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
